In [1]:
import numpy as np
import msicpe.san as san
from plotly import express as px
import scipy.signal as signal

import pandas as pd

import sounddevice as sd # pour jouer les sons inconnus aux signaux

In [2]:
def trace(t, courbes, titre, nom_abscisse="t"):
    # Trace plusieurs courbes (dict nom -> tableau) dans le même repère
    df = pd.DataFrame({nom_abscisse: t, **courbes})
    fig = px.line(df, x=nom_abscisse, y=list(courbes.keys()), title=titre)

    fig.show()

In [3]:
def extract_signal(nom):
    signal_dict = np.load(nom)  # signal_dict est un dictionnaire, → contenant les données contenues dans le fichier signal.npz
    s = signal_dict['s']  # extraction du signal utile
    t = signal_dict['t']  # extraction du vecteur temps associé au signal
    return s, t

# 1 Partie 1 - Introduction à l’analyse spectrale

In [4]:
s1,t = extract_signal("data_1/piano_G.npz")
trace(t, {"s": s1}, "Signal piano_G.npz", nom_abscisse="Temps (s)")

s2,t = extract_signal("data_1/tuning_fork_440Hz.npz")
trace(t, {"s": s2}, "Signal tuning_fork_440Hz.npz", nom_abscisse="Temps (s)")

s3,t = extract_signal("data_1/violin_C.npz")
trace(t, {"s": s3}, "Signal violin_C.npz", nom_abscisse="Temps (s)")



# 2 Partie 2 - Reconnaissance automatique de voyelle


## 2.1 Lecture des fichiers et affichages

In [5]:
a1,t = extract_signal("data_2/a_1.npz")
# trace(t, {"s": a1}, "Signal a_1.npz", nom_abscisse="Temps (s)")

e1,t = extract_signal("data_2/e_1.npz")
# trace(t, {"s": e1}, "Signal e_1.npz", nom_abscisse="Temps (s)")

i1,t = extract_signal("data_2/i_1.npz")
# trace(t, {"s": i1}, "Signal i_1.npz", nom_abscisse="Temps (s)")


trace(t, {"a_1": a1, "e_1": e1, "i_1": i1}, "Comparaison des signaux", nom_abscisse="Temps (s)")

In [6]:
# help(san.trans_fourier)

nu,tf_a1=san.trans_fourier(a1, t)
# trace(nu, {"a_1": np.real(tf_a1)}, "Transformée de Fourier du signal a_1", nom_abscisse="Fréquence (Hz)")

nu,tf_e1=san.trans_fourier(e1, t)
# trace(nu, {"e_1": np.real(tf_e1)}, "Transformée de Fourier du signal e_1", nom_abscisse="Fréquence (Hz)")

nu,tf_i1=san.trans_fourier(i1, t)
# trace(nu, {"i_1": np.real(tf_i1)}, "Transformée de Fourier du signal i_1", nom_abscisse="Fréquence (Hz)")

trace(nu, {"a_1": np.real(tf_a1), "e_1": np.real(tf_e1), "i_1": np.real(tf_i1)}, "Comparaison des transformées de Fourier", nom_abscisse="Fréquence (Hz)")


In [7]:
nu,tf_a1=san.trans_fourier(a1, t)
# trace(nu, {"a_1": np.log10(np.real(tf_a1))}, "Transformée de Fourier du signal a_1 en base10", nom_abscisse="Fréquence (Hz)")

nu,tf_e1=san.trans_fourier(e1, t)
# trace(nu, {"e_1": np.log10(np.real(tf_e1))}, "Transformée de Fourier du signal e_1 en base10", nom_abscisse="Fréquence (Hz)")

nu,tf_i1=san.trans_fourier(i1, t)
# trace(nu, {"i_1": np.log10(np.real(tf_i1))}, "Transformée de Fourier du signal i_1 en base10", nom_abscisse="Fréquence (Hz)")

trace(nu, {"a_1": np.log10(np.real(tf_a1)), "e_1": np.log10(np.real(tf_e1)), "i_1": np.log10(np.real(tf_i1))}, "Comparaison des transformées de Fourier en base10", nom_abscisse="Fréquence (Hz)")


C:\Users\darkt\AppData\Local\Temp\ipykernel_15448\932385934.py:10: RuntimeWarning: invalid value encountered in log10
  trace(nu, {"a_1": np.log10(np.real(tf_a1)), "e_1": np.log10(np.real(tf_e1)), "i_1": np.log10(np.real(tf_i1))}, "Comparaison des transformées de Fourier en base10", nom_abscisse="Fréquence (Hz)")


## 2.2 Apprentissage − Caractérisation des voyelles

In [8]:
help(san.detect_fondamentale)
help(san.detect_pics)

a1,t = extract_signal("data_2/a_1.npz")
a2,t = extract_signal("data_2/a_2.npz")
a3,t = extract_signal("data_2/a_3.npz")
a4,t = extract_signal("data_2/a_4.npz")
a5,t = extract_signal("data_2/a_5.npz")

a6,t = extract_signal("data_2/a_6.npz")
a7,t = extract_signal("data_2/a_7.npz")

e1,t = extract_signal("data_2/e_1.npz")
e2,t = extract_signal("data_2/e_2.npz")
e3,t = extract_signal("data_2/e_3.npz")
e4,t = extract_signal("data_2/e_4.npz")
e5,t = extract_signal("data_2/e_5.npz")

e6,t = extract_signal("data_2/e_6.npz")
e7,t = extract_signal("data_2/e_7.npz")

i1,t = extract_signal("data_2/i_1.npz")
i2,t = extract_signal("data_2/i_2.npz")
i3,t = extract_signal("data_2/i_3.npz")
i4,t = extract_signal("data_2/i_4.npz")
i5,t = extract_signal("data_2/i_5.npz")

i6,t = extract_signal("data_2/i_6.npz")
i7,t = extract_signal("data_2/i_7.npz")

o1,t = extract_signal("data_2/o_1.npz")
o2,t = extract_signal("data_2/o_2.npz")
o3,t = extract_signal("data_2/o_3.npz")
o4,t = extract_signal("data_2/o_4.npz")
o5,t = extract_signal("data_2/o_5.npz")

o6,t = extract_signal("data_2/o_6.npz")
o7,t = extract_signal("data_2/o_7.npz")


u1,t = extract_signal("data_2/u_1.npz")
u2,t = extract_signal("data_2/u_2.npz")
u3,t = extract_signal("data_2/u_3.npz")
u4,t = extract_signal("data_2/u_4.npz")
u5,t = extract_signal("data_2/u_5.npz")

u6,t = extract_signal("data_2/u_6.npz")
u7,t = extract_signal("data_2/u_7.npz")

Help on function detect_fondamentale in module msicpe.san.detect_fondamentale:

detect_fondamentale(spectre, nu, threshold=0.25)
    Fonction permettant de déterminer la fréquence fondamentale d'un signal à partir de son spectre.
    Args:
        spectre (ndarray): spectre du signal
        nu (ndarray): vecteur des fréquences associées au spectre
        threshold (float, optional): seuil pour la détection de la fréquence fondamentale. Default à 0.25.
    Returns:
        nu_fond (float): fréquence fondamentale détectée en Hz

Help on function detect_pics in module msicpe.san.detect_pics:

detect_pics(spectre, freq_list, nu)
    Fonction permettant de détecter les pics harmoniques dans un spectre.
    Args:
        spectre (ndarray): spectre du signal
        freq_list (list): liste des fréquences harmoniques à détecter
        nu (ndarray): vecteur des fréquences associées au spectre
    Returns:
        freq_peaks (ndarray): fréquences des pics détectés
        peak_amplitudes (nda

In [9]:
H = 30

sons_a = [a1, a2, a3, a4, a5]
sons_e = [e1, e2, e3, e4, e5]
sons_i = [i1, i2, i3, i4, i5]
sons_o = [o1, o2, o3, o4, o5]
sons_u = [u1, u2, u3, u4, u5]


def vecteur_caracteristique(signals,t,H):
    vecteurs = []
    for son in signals:
        nu,tf=san.trans_fourier(son-np.mean(son), t) # on recentre le signal pour avoir une moyenne nulle
        spectre = np.abs(tf) / np.max(np.abs(tf)) # indication prof
        
        f0 = san.detect_fondamentale(
            spectre, nu, 0.25
        )

        # H+1 harmoniques : f0, 2*f0, ..., (H+1)*f0
        freq_list = [(k + 1) * f0 for k in range(H + 1)]

        freq_peaks, peak_amplitudes= san.detect_pics(spectre, freq_list, nu)
        vecteur = np.log10(np.maximum(peak_amplitudes, 1e-12))
        vecteurs.append(vecteur)
    
    return np.array(vecteurs)

def valeur_moyenne(vecteurs):
    return np.mean(vecteurs, axis=0) # Ne sert à rien de faire une fonction

vecteurs_a = vecteur_caracteristique(sons_a,t,H)
vecteurs_e = vecteur_caracteristique(sons_e,t,H)
vecteurs_i = vecteur_caracteristique(sons_i,t,H)
vecteurs_o = vecteur_caracteristique(sons_o,t,H)
vecteurs_u = vecteur_caracteristique(sons_u,t,H)

moyenne_a = valeur_moyenne(vecteurs_a)
moyenne_e = valeur_moyenne(vecteurs_e)
moyenne_i = valeur_moyenne(vecteurs_i)
moyenne_o = valeur_moyenne(vecteurs_o)
moyenne_u = valeur_moyenne(vecteurs_u)

# print(moyenne_a)
# print(moyenne_e)
# print(moyenne_i)
# print(moyenne_o)
# print(moyenne_u)



harmoniques = np.arange(0, H+1)  # 1, 2, ..., H

trace(harmoniques, {"a": moyenne_a, "a1":vecteurs_a[0], "a2":vecteurs_a[1], "a3":vecteurs_a[2], "a4":vecteurs_a[3], "a5":vecteurs_a[4]}, "", nom_abscisse="Harmoniques")
# trace(harmoniques, {"e": moyenne_e, "e1":vecteurs_e[0], "e2":vecteurs_e[1], "e3":vecteurs_e[2], "e4":vecteurs_e[3], "e5":vecteurs_e[4]}, "", nom_abscisse="Harmoniques")
# trace(harmoniques, {"i": moyenne_i, "i1":vecteurs_i[0], "i2":vecteurs_i[1], "i3":vecteurs_i[2], "i4":vecteurs_i[3], "i5":vecteurs_i[4]}, "", nom_abscisse="Harmoniques")
# trace(harmoniques, {"o": moyenne_o, "o1":vecteurs_o[0], "o2":vecteurs_o[1], "o3":vecteurs_o[2], "o4":vecteurs_o[3], "o5":vecteurs_o[4]}, "", nom_abscisse="Harmoniques")
# trace(harmoniques, {"u": moyenne_u, "u1":vecteurs_u[0], "u2":vecteurs_u[1], "u3":vecteurs_u[2], "u4":vecteurs_u[3], "u5":vecteurs_u[4]}, "", nom_abscisse="Harmoniques")



# trace(harmoniques, {"a": moyenne_a, "e": moyenne_e, "i": moyenne_i, "o": moyenne_o, "u": moyenne_u}, "", nom_abscisse="Harmoniques")





In [10]:
sons_a_test = [a6, a7]
sons_e_test = [e6, e7]
sons_i_test = [i6, i7]
sons_o_test = [o6, o7]
sons_u_test = [u6, u7]

vecteurs_a_test = vecteur_caracteristique(sons_a_test,t,H)
vecteurs_e_test = vecteur_caracteristique(sons_e_test,t,H)
vecteurs_i_test = vecteur_caracteristique(sons_i_test,t,H)
vecteurs_o_test = vecteur_caracteristique(sons_o_test,t,H)
vecteurs_u_test = vecteur_caracteristique(sons_u_test,t,H)


def voyelles_prediction(vecteurs_test):

    moyennes=[moyenne_a, moyenne_e, moyenne_i, moyenne_o, moyenne_u]

    voyelles = []
    
    for k in range(len(vecteurs_test)):                 # k parcourt les sons de test (combien y en a-t-il ?)
        distances = []
        for i in range(len(moyennes)):             # i parcourt les 5 voyelles
            d = np.sqrt(np.sum((vecteurs_test[k] - moyennes[i])**2))                      # distance euclidienne entre vecteurs_a_test[k] et moyennes[i]
            distances.append(d)

        min_index = np.argmin(distances)  # on récupère la distance la plus proche
        voyelles.append("aeiou"[min_index])  # on ajoute l'indice de la voyelle la plus proche

    return voyelles

tableau_a = voyelles_prediction(vecteurs_a_test)
tableau_e = voyelles_prediction(vecteurs_e_test)
tableau_i = voyelles_prediction(vecteurs_i_test)
tableau_o = voyelles_prediction(vecteurs_o_test)
tableau_u = voyelles_prediction(vecteurs_u_test)

print("Prédictions pour les voyelles a:", tableau_a)
print("Prédictions pour les voyelles e:", tableau_e)
print("Prédictions pour les voyelles i:", tableau_i)
print("Prédictions pour les voyelles o:", tableau_o)
print("Prédictions pour les voyelles u:", tableau_u)


Prédictions pour les voyelles a: ['a', 'a']
Prédictions pour les voyelles e: ['e', 'e']
Prédictions pour les voyelles i: ['i', 'i']
Prédictions pour les voyelles o: ['o', 'o']
Prédictions pour les voyelles u: ['u', 'u']


In [11]:
s1,t = extract_signal("data_3/signal_1.npz")
s2,t = extract_signal("data_3/signal_2.npz")
s3,t = extract_signal("data_3/signal_3.npz")
s4,t = extract_signal("data_3/signal_4.npz")
s5,t = extract_signal("data_3/signal_5.npz")
s6,t = extract_signal("data_3/signal_6.npz")
s7,t = extract_signal("data_3/signal_7.npz")
s8,t = extract_signal("data_3/signal_8.npz")
s9,t = extract_signal("data_3/signal_9.npz")
s10,t = extract_signal("data_3/signal_10.npz")
s11,t = extract_signal("data_3/signal_11.npz")
s12,t = extract_signal("data_3/signal_12.npz")
s13,t = extract_signal("data_3/signal_13.npz")
s14,t = extract_signal("data_3/signal_14.npz")
s15,t = extract_signal("data_3/signal_15.npz")

sons_a_test = [s1, s2, s3, s4, s5, s6, s7, s8, s9, s10, s11, s12, s13, s14, s15]

vecteurs_inconnus_test = vecteur_caracteristique(sons_a_test,t,H)

voyelles_inconnus = voyelles_prediction(vecteurs_inconnus_test)

print("Prédictions pour les voyelles inconnues:", voyelles_inconnus)


Prédictions pour les voyelles inconnues: ['e', 'i', 'i', 'a', 'e', 'o', 'a', 'u', 'o', 'a', 'u', 'u', 'o', 'i', 'e']


In [12]:
s, t = extract_signal("data_3/signal_15.npz")
fe = 1 / (t[1] - t[0])                 # 12 000 Hz
s = s.flatten()
sd.play(s / np.max(np.abs(s)), fe)     # normalisé entre -1 et 1 pour éviter la saturation
sd.wait()                              # attend la fin avant de passer au suivant

In [13]:




H_possibles = [5, 10, 15, 20, 25, 30, 32, 35, 37, 40]

for H in H_possibles:
    vecteurs_a = vecteur_caracteristique(sons_a,t,H)
    vecteurs_e = vecteur_caracteristique(sons_e,t,H)
    vecteurs_i = vecteur_caracteristique(sons_i,t,H)
    vecteurs_o = vecteur_caracteristique(sons_o,t,H)
    vecteurs_u = vecteur_caracteristique(sons_u,t,H)

    moyenne_a = valeur_moyenne(vecteurs_a)
    moyenne_e = valeur_moyenne(vecteurs_e)
    moyenne_i = valeur_moyenne(vecteurs_i)
    moyenne_o = valeur_moyenne(vecteurs_o)
    moyenne_u = valeur_moyenne(vecteurs_u)

    # On calcule la dispersion des vecteurs par rapport à la moyenne pour chaque voyelle
    # il faut que la dispersion soit la plus faible possible pour que les vecteurs soient les plus proches possible de la moyenne, donc que le choix de H soit le meilleur possible
    dispersion_a = np.mean(np.sqrt(np.sum((vecteurs_a - moyenne_a) ** 2, axis=1)))
    dispersion_e = np.mean(np.sqrt(np.sum((vecteurs_e - moyenne_e) ** 2, axis=1)))
    dispersion_i = np.mean(np.sqrt(np.sum((vecteurs_i - moyenne_i) ** 2, axis=1)))
    dispersion_o = np.mean(np.sqrt(np.sum((vecteurs_o - moyenne_o) ** 2, axis=1)))
    dispersion_u = np.mean(np.sqrt(np.sum((vecteurs_u - moyenne_u) ** 2, axis=1)))

    moyennes=[moyenne_a, moyenne_e, moyenne_i, moyenne_o, moyenne_u]
    dispersions=[dispersion_a, dispersion_e, dispersion_i, dispersion_o, dispersion_u]

    R_min = np.inf
    couple_critique = None

    for i in range(len(moyennes)):
        for j in range(len(moyennes)):
            if i != j:
                R = (np.sqrt(np.sum((moyennes[i] - moyennes[j]) ** 2))) / (dispersions[i] + dispersions[j])
                if R < R_min:
                    R_min = R
                    couple_critique = ("aeiou"[i], "aeiou"[j])

    # et que les moyennes soit les plus eloignées possible les unes des autres, donc que le choix de H soit le meilleur possible


    print(f"{H} : R_min = {R_min} couple critique : {couple_critique}")

5 : R_min = 0.34413936734199524 couple critique : ('i', 'u')
10 : R_min = 0.5653736591339111 couple critique : ('i', 'u')
15 : R_min = 0.6478608250617981 couple critique : ('i', 'u')
20 : R_min = 0.596733033657074 couple critique : ('i', 'u')
25 : R_min = 0.7065098881721497 couple critique : ('i', 'u')
30 : R_min = 0.8595445156097412 couple critique : ('a', 'e')
32 : R_min = 0.8432077169418335 couple critique : ('a', 'e')
35 : R_min = 0.8283219933509827 couple critique : ('a', 'e')
37 : R_min = 0.8080519437789917 couple critique : ('a', 'e')
40 : R_min = 0.8006390333175659 couple critique : ('a', 'e')


In [14]:
H = 30

vecteurs_a = vecteur_caracteristique(sons_a, t, H)
vecteurs_e = vecteur_caracteristique(sons_e, t, H)
vecteurs_i = vecteur_caracteristique(sons_i, t, H)
vecteurs_o = vecteur_caracteristique(sons_o, t, H)
vecteurs_u = vecteur_caracteristique(sons_u, t, H)

moyenne_a = valeur_moyenne(vecteurs_a)
moyenne_e = valeur_moyenne(vecteurs_e)
moyenne_i = valeur_moyenne(vecteurs_i)
moyenne_o = valeur_moyenne(vecteurs_o)
moyenne_u = valeur_moyenne(vecteurs_u)

harmoniques = np.arange(0, H + 1)

PARTIE PREPARATION POWERPOINT

In [15]:
nu,S3= san.trans_fourier(s3, t)

trace(nu, {"S3": np.real(S3)}, "Transformée de Fourier réel du signal s3", nom_abscisse="Fréquence (Hz)")
trace(nu, {"S3": np.imag(S3)}, "Transformée de Fourier imaginaire du signal s3", nom_abscisse="Fréquence (Hz)")

trace(t, {"s3": s3}, "Tracer temporel du signal s3", nom_abscisse="Temps (s)")

In [16]:
nu, A_1 = san.trans_fourier(a1, t)
nu, E_1 = san.trans_fourier(e1, t)
nu, I_1 = san.trans_fourier(i1, t)
nu, O_1 = san.trans_fourier(o1, t)
nu, U_1 = san.trans_fourier(u1, t)

trace(nu, {"A_1": np.abs(A_1)}, "Spectre du signal a_1", nom_abscisse="Fréquence (Hz)")
trace(nu, {"E_1": np.abs(E_1)}, "Spectre du signal e_1", nom_abscisse="Fréquence (Hz)")
trace(nu, {"I_1": np.abs(I_1)}, "Spectre du signal i_1", nom_abscisse="Fréquence (Hz)")
trace(nu, {"O_1": np.abs(O_1)}, "Spectre du signal o_1", nom_abscisse="Fréquence (Hz)")
trace(nu, {"U_1": np.abs(U_1)}, "Spectre du signal u_1", nom_abscisse="Fréquence (Hz)")

In [17]:

nu, A = san.trans_fourier(moyenne_a, t)
nu, E = san.trans_fourier(moyenne_e, t)
nu, I = san.trans_fourier(moyenne_i, t)
nu, O = san.trans_fourier(moyenne_o, t)
nu, U = san.trans_fourier(moyenne_u, t)


#trace(nu, {"A": np.abs(A)}, "Spectre de la moyenne des signaux a", nom_abscisse="Fréquence (Hz)")
#trace(nu, {"E": np.abs(E)}, "Spectre de la moyenne des signaux e", nom_abscisse="Fréquence (Hz)")
#trace(nu, {"I": np.abs(I)}, "Spectre de la moyenne des sign aux i", nom_abscisse="Fréquence (Hz)")
#trace(nu, {"O": np.abs(O)}, "Spectre de la moyenne des signaux o", nom_abscisse="Fréquence (Hz)")
#trace(nu, {"U": np.abs(U)}, "Spectre de la moyenne des signaux u", nom_abscisse="Fréquence (Hz)")

trace(harmoniques, {"moyenne_a": moyenne_a, "moyenne_e": moyenne_e, "moyenne_i": moyenne_i, "moyenne_o": moyenne_o, "moyenne_u": moyenne_u}, "Comparaison des moyennes des signaux", nom_abscisse="Harmoniques")


#On a prit H= 30, car c'est la valeur ou il y a le moins de dispersion entre les voyelles, donc c'est la valeur qui permet de mieux différencier les voyelles entre elles.
#Pour la méthode de normalisation, on a prit la normalisation en base 10, car c'est la méthode qui permet de mieux différencier les voyelles entre elles, car elle permet de mieux visualiser les différences entre les harmoniques des voyelles.

In [ ]:
H = 30
harmoniques = np.arange(0, H + 1)
vecteur_s3 = vecteur_caracteristique([s3], t, H)[0]

trace(harmoniques, {"s3": vecteur_s3, "moyenne_i": moyenne_i},"Enveloppe des harmoniques : s3 vs moyenne de i", nom_abscisse="Harmoniques")


correlation = np.corrcoef(vecteur_s3, moyenne_i)[0, 1]
print("Corrélation entre s3 et moyenne_i :", correlation)

sons_a_test = [s3]
vecteurs_inconnus_test = vecteur_caracteristique(sons_a_test, t, H)
voyelles_inconnus = voyelles_prediction(vecteurs_inconnus_test)

print("Prédictions pour la voyelle inconnue :", voyelles_inconnus)

moyennes = [moyenne_a, moyenne_e, moyenne_i, moyenne_o, moyenne_u]
for i in range(len(moyennes)):
    d = np.sqrt(np.sum((vecteur_s3 - moyennes[i]) ** 2))
    print("Distance entre s3 et la moyenne de", "aeiou"[i], ":", d)

#La distance euclidienne entre le vecteur caractéristique de iet  de la voyelle inconnue est de 1.5, c'est la distance la plus faible, donc la voyelle inconnue est bien un i. 
#On remaraque en corrélant s3 et i_moyen que la corrélation vaut 0.93, alors les deux signaix sont tres similaire, donc la voyelle inconnue est bien un i.

Corrélation entre s3 et moyenne_i : 0.9322290518217867
Prédictions pour la voyelle inconnue : ['i']
Distance entre s3 et la moyenne de a : 7.0362577
Distance entre s3 et la moyenne de e : 4.9715767
Distance entre s3 et la moyenne de i : 1.5440917
Distance entre s3 et la moyenne de o : 3.462933
Distance entre s3 et la moyenne de u : 3.3429973
